<a href="https://colab.research.google.com/github/sukritghosh886-hub/Tower--of-hanoi/blob/main/Indian_house_pradication.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

pip install pandas

In [ ]:
# 1. Install the Kaggle library
!pip install kaggle

# 2. Upload your Kaggle API key (kaggle.json)
#    - Go to Kaggle (kaggle.com), click on your profile picture, then 'Account'.
#    - Scroll down to the 'API' section and click 'Create New API Token'.
#    - Download the 'kaggle.json' file.
#    - In Colab, click the folder icon on the left sidebar, then the upload icon.
#    - Upload 'kaggle.json' to the current directory (e.g., /content/).

# 3. Make sure the kaggle.json is in the correct directory and has the right permissions
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# 4. Download the dataset
# The dataset ID is 'ankushpandey77/india-house-price-prediction'
!kaggle datasets download -d ankushpandey77/india-house-price-prediction

# 5. Unzip the dataset (if it's a zip file)
!unzip india-house-price-prediction.zip

# You should now see 'india_housing_prices.csv' in your files.

cp: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory
403 Client Error: Forbidden for url: https://api.kaggle.com/v1/datasets.DatasetApiService/GetDatasetMetadata
unzip:  cannot find or open india-house-price-prediction.zip, india-house-price-prediction.zip.zip or india-house-price-prediction.zip.ZIP.


In [ ]:

import pandas as pd

# 1. Creating dummy data so the code runs perfectly in Colab right now
raw_data = {
    'City': ['Mumbai', 'Delhi', 'Bangalore', 'Chennai', 'Kolkata', 'Dehradun'],
    'Property_Type': ['Apartment', 'Independent House', 'Apartment', 'Apartment', 'Independent House', 'Apartment'],
    'BHK': [2, 4, 3, 2, 3, 2],
    'Bathrooms': [2, 4, 2, 2, 3, 1],
    'Price_in_Lakhs': [120, 350, 95, 80, 110, 65]
}
df = pd.DataFrame(raw_data)

# 2. Add "Climate" by mapping Indian cities to their general climate zones
climate_zones = {
    'Mumbai': 'Humid Tropical',
    'Delhi': 'Composite / Extreme Hot & Cold',
    'Bangalore': 'Moderate / Tropical Savanna',
    'Chennai': 'Tropical Wet & Dry',
    'Kolkata': 'Tropical Wet',
    'Dehradun': 'Subtropical Highland / Cool'
}
df['Climate'] = df['City'].map(climate_zones).fillna('Warm & Humid')

# 3. Add "Rooms" and "Toilets"
df['Total_Rooms'] = df['BHK']
df['Toilets'] = df['Bathrooms']

# 4. Add "Kitchens"
df['Kitchens'] = df['Property_Type'].apply(lambda x: 2 if x == 'Independent House' else 1)

# 5. Calculate "Max_People" (Capacity)
df['Max_People'] = (df['BHK'] * 2) + 1

# Preview your custom dataset
df[['City', 'Price_in_Lakhs', 'Climate', 'Total_Rooms', 'Toilets', 'Kitchens', 'Max_People']]

,City,Price_in_Lakhs,Climate,Total_Rooms,Toilets,Kitchens,Max_People
0,Mumbai,120,Humid Tropical,2,2,1,5
1,Delhi,350,Composite / Extreme Hot & Cold,4,4,2,9
2,Bangalore,95,Moderate / Tropical Savanna,3,2,1,7
3,Chennai,80,Tropical Wet & Dry,2,2,1,5
4,Kolkata,110,Tropical Wet,3,3,2,7
5,Dehradun,65,Subtropical Highland / Cool,2,1,1,5


In [ ]:

# 1. Choose your preferred climate (e.g., 'Moderate / Tropical Savanna' for Bangalore)
ideal_climate = df[df['Climate'] == 'Moderate / Tropical Savanna']

# 2. Filter for houses that fit enough people (e.g., 5 or more) and have at least 2 toilets
spacious_houses = ideal_climate[(ideal_climate['Max_People'] >= 5) & (ideal_climate['Toilets'] >= 2)]

# 3. Sort the results by Price to find the cheapest (best price) at the top
best_deals = spacious_houses.sort_values(by='Price_in_Lakhs', ascending=True)

print("--- BEST SPECIFIC HOUSE DEALS ---")
print(best_deals[['City', 'Property_Type', 'Price_in_Lakhs', 'Climate', 'Max_People']])

--- BEST SPECIFIC HOUSE DEALS ---
        City Property_Type  Price_in_Lakhs                      Climate  \
2  Bangalore     Apartment              95  Moderate / Tropical Savanna   

   Max_People  
2           7  


In [ ]:

# Group the data by City and calculate the averages
city_analysis = df.groupby('City').agg({
    'Price_in_Lakhs': 'mean',  # Finds the average cost of a house in that city
    'Max_People': 'mean',      # Finds the average capacity
    'Kitchens': 'max'          # Finds the maximum number of kitchens available
}).reset_index()

# Sort the cities from cheapest average price to most expensive
best_cities_ranked = city_analysis.sort_values(by='Price_in_Lakhs', ascending=True)

print("\n--- CITIES RANKED BY AVERAGE AFFORDABILITY ---")
print(best_cities_ranked)


--- CITIES RANKED BY AVERAGE AFFORDABILITY ---
        City  Price_in_Lakhs  Max_People  Kitchens
2   Dehradun            65.0         5.0         1
1    Chennai            80.0         5.0         1
0  Bangalore            95.0         7.0         1
4    Kolkata           110.0         7.0         2
5     Mumbai           120.0         5.0         1
3      Delhi           350.0         9.0         2


In [ ]:

from google.colab import files
uploaded = files.upload()
df = pd.read_csv(next(iter(uploaded)))

Saving india_housing_prices_massive.csv to india_housing_prices_massive.csv


In [ ]:

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

# 1. LOAD THE MASSIVE DATASET
df = pd.read_csv("india_housing_prices_massive.csv")

# 2. FEATURE ENGINEERING (Adding the columns we made earlier)
climate_zones = {
    'Mumbai': 'Humid Tropical', 'Delhi': 'Composite / Extreme Hot & Cold',
    'Bangalore': 'Moderate / Tropical Savanna', 'Chennai': 'Tropical Wet & Dry',
    'Kolkata': 'Tropical Wet', 'Dehradun': 'Subtropical Highland / Cool'
}
df['Climate'] = df['City'].map(climate_zones).fillna('Warm & Humid')
df['Total_Rooms'] = df['BHK']
df['Toilets'] = df['Bathrooms']
df['Kitchens'] = df['Property_Type'].apply(lambda x: 2 if x in ['Independent House', 'Villa'] else 1)
df['Max_People'] = (df['BHK'] * 2) + 1

# 3. ENCODE AND SPLIT
df_encoded = pd.get_dummies(df, columns=['City', 'Property_Type', 'Climate'], drop_first=True)
y = df_encoded['Price_in_Lakhs']
X = df_encoded.drop('Price_in_Lakhs', axis=1)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. TRAIN AND TEST THE MODEL
model = RandomForestRegressor(random_state=42)
model.fit(X_train, y_train)
predictions = model.predict(X_test)
error = mean_absolute_error(y_test, predictions)

print(f"Model trained on 5,000 rows! Average prediction error is {error:.2f} Lakhs.")

Model trained on 5,000 rows! Average prediction error is 13.84 Lakhs.


In [ ]:

import pandas as pd
import numpy as np

# Generate 5,000 rows of logical fake data
np.random.seed(42)
num_rows = 5000

cities = ['Mumbai', 'Delhi', 'Bangalore', 'Chennai', 'Kolkata', 'Dehradun']
property_types = ['Apartment', 'Independent House', 'Villa']
city_base_price = {'Mumbai': 120, 'Delhi': 85, 'Bangalore': 70, 'Chennai': 55, 'Kolkata': 40, 'Dehradun': 45}

city_col = np.random.choice(cities, num_rows, p=[0.25, 0.2, 0.2, 0.15, 0.1, 0.1])
prop_type_col = np.random.choice(property_types, num_rows, p=[0.7, 0.2, 0.1])
bhk_col = np.random.choice([1, 2, 3, 4, 5], num_rows, p=[0.1, 0.4, 0.3, 0.15, 0.05])
bath_col = np.clip(bhk_col + np.random.randint(-1, 2, size=num_rows), 1, 6)

prices = []
for i in range(num_rows):
    city, prop, bhk, bath = city_col[i], prop_type_col[i], bhk_col[i], bath_col[i]
    price = city_base_price[city] + (bhk * 20) + (bath * 8)
    if prop == 'Villa': price *= 2.0
    elif prop == 'Independent House': price *= 1.4
    price *= np.random.uniform(0.85, 1.15)
    prices.append(round(price, 2))

df = pd.DataFrame({'City': city_col, 'Property_Type': prop_type_col, 'BHK': bhk_col, 'Bathrooms': bath_col, 'Price_in_Lakhs': prices})
print("Dataset created with 5,000 rows!")

Dataset created with 5,000 rows!


In [ ]:

import pandas as pd
import io

# The full CSV content (copied from your prompt)
csv_data = """City,Property_Type,BHK,Bathrooms,Price_in_Lakhs
Delhi,Apartment,2,2,132.56
Dehradun,Apartment,2,1,101.3
Chennai,Independent House,2,1,151.68
... (all 10,000+ rows go here)
"""

# Load into DataFrame
df = pd.read_csv(io.StringIO(csv_data))

# Quick preview
print("✅ Dataset loaded!")
print(f"Shape: {df.shape}")
print("\nFirst 5 rows:")
print(df.head())

# Basic statistics
print("\n📊 Summary statistics:")
print(df.describe())

# Check for missing values
print("\n🔍 Missing values:")
print(df.isnull().sum())

✅ Dataset loaded!
Shape: (4, 5)

First 5 rows:
          City       Property_Type  BHK  Bathrooms  Price_in_Lakhs
0        Delhi           Apartment  2.0        2.0          132.56
1     Dehradun           Apartment  2.0        1.0          101.30
2      Chennai   Independent House  2.0        1.0          151.68
3  ... (all 10  000+ rows go here)  NaN        NaN             NaN

📊 Summary statistics:
       BHK  Bathrooms  Price_in_Lakhs
count  3.0   3.000000        3.000000
mean   2.0   1.333333      128.513333
std    0.0   0.577350       25.432612
min    2.0   1.000000      101.300000
25%    2.0   1.000000      116.930000
50%    2.0   1.000000      132.560000
75%    2.0   1.500000      142.120000
max    2.0   2.000000      151.680000

🔍 Missing values:
City              0
Property_Type     0
BHK               1
Bathrooms         1
Price_in_Lakhs    1
dtype: int64


In [ ]:

# Import the required machine learning tools
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

print("Starting Machine Learning Pipeline...\n")

# --- FEATURE ENGINEERING (Adding the columns we made earlier) ---
# This step creates the 'Climate' and other derived features that the model needs.
climate_zones = {
    'Mumbai': 'Humid Tropical', 'Delhi': 'Composite / Extreme Hot & Cold',
    'Bangalore': 'Moderate / Tropical Savanna', 'Chennai': 'Tropical Wet & Dry',
    'Kolkata': 'Tropical Wet', 'Dehradun': 'Subtropical Highland / Cool'
}
df['Climate'] = df['City'].map(climate_zones).fillna('Warm & Humid')
df['Total_Rooms'] = df['BHK']
df['Toilets'] = df['Bathrooms']
df['Kitchens'] = df['Property_Type'].apply(lambda x: 2 if x in ['Independent House', 'Villa'] else 1)
df['Max_People'] = (df['BHK'] * 2) + 1
print("Feature Engineering Complete.")

# --- STEP 1: Convert Text to Numbers (Encoding) ---
# Machine learning models only understand numbers. This converts text columns into 1s and 0s.
df_encoded = pd.get_dummies(df, columns=['City', 'Property_Type', 'Climate'], drop_first=True)
print("Step 1 Complete: Text converted to numbers.")

# --- STEP 2: Separate Features from the Target ---
# y is the target (what we want to predict)
y = df_encoded['Price_in_Lakhs']

# X is the features (the clues used to predict the price)
X = df_encoded.drop('Price_in_Lakhs', axis=1)
print("Step 2 Complete: Features (X) and Target (y) separated.")

# --- STEP 3: Split into Training and Testing Data ---
# 80% of the data goes to training, 20% is held back for testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Step 3 Complete: Data split into 80% training and 20% testing.")

# --- STEP 4: Train the Predictive Model ---
# Initialize the Random Forest Regressor
model = RandomForestRegressor(random_state=42)

# Train the model using the training data
model.fit(X_train, y_train)

# Ask the model to predict prices for the 20% of data it hasn't seen
predictions = model.predict(X_test)

# Compare the model's predictions to the actual real prices
error = mean_absolute_error(y_test, predictions)
print("Step 4 Complete: Model successfully trained.\n")

# --- FINAL RESULTS ---
print("--- MODEL ACCURACY ---")
print(f"On average, the model's predictions are off by: {error:.2f} Lakhs.")

Starting Machine Learning Pipeline...

Feature Engineering Complete.
Step 1 Complete: Text converted to numbers.
Step 2 Complete: Features (X) and Target (y) separated.
Step 3 Complete: Data split into 80% training and 20% testing.
Step 4 Complete: Model successfully trained.

--- MODEL ACCURACY ---
On average, the model's predictions are off by: 13.84 Lakhs.


In [ ]:

import pandas as pd

# 1. Type in the features of your brand new house
# Make sure the text exactly matches the categories in your dataset!
new_house_data = {
    'City': ['Delhi'],
    'Property_Type': ['Independent House'],
    'BHK': [4],
    'Bathrooms': [4],
    'Climate': ['Composite / Extreme Hot & Cold'], # Delhi's climate from our mapping
    'Total_Rooms': [4],
    'Toilets': [4],
    'Kitchens': [2],
    'Max_People': [9]   # (4 BHK * 2) + 1
}

# 2. Convert your single house into a Pandas DataFrame
new_house_df = pd.DataFrame(new_house_data)

# 3. Convert the text to numbers (Encoding)
new_house_encoded = pd.get_dummies(new_house_df)

# 4. The Magic Trick: Align the columns
# Your new house only has "Delhi", so get_dummies won't create a column for "Mumbai".
# This step forces the new data to have the exact same columns as your training data (X),
# filling in any missing cities or property types with 0s.
new_house_ready = new_house_encoded.reindex(columns=X.columns, fill_value=0)

# 5. Make the prediction!
predicted_price = model.predict(new_house_ready)

print("--- PREDICTION RESULT ---")
print(f"The estimated price for this house is: {predicted_price[0]:.2f} Lakhs")

--- PREDICTION RESULT ---
The estimated price for this house is: 255.21 Lakhs


In [ ]:

import joblib

# 1. Save the trained model
joblib.dump(model, 'house_price_predictor.pkl')

# 2. Save the exact column names the model learned from
# This is crucial so your model remembers the exact order of the columns (like Mumbai vs Delhi)
joblib.dump(X.columns, 'model_columns.pkl')

print("Model successfully saved to disk!")

Model successfully saved to disk!


In [ ]:

import pandas as pd
import joblib

# 1. Load your trained model and the column names
loaded_model = joblib.load('house_price_predictor.pkl')
trained_columns = joblib.load('model_columns.pkl')

# 2. Create your brand new house data
new_house_data = {
    'City': ['Bangalore'],
    'Property_Type': ['Apartment'],
    'BHK': [3],
    'Bathrooms': [2],
    'Climate': ['Moderate / Tropical Savanna'],
    'Total_Rooms': [3],
    'Toilets': [2],
    'Kitchens': [1],
    'Max_People': [7]
}

# 3. Process it exactly like before
new_house_df = pd.DataFrame(new_house_data)
new_house_encoded = pd.get_dummies(new_house_df)

# 4. Use the saved columns to perfectly align the new data
new_house_ready = new_house_encoded.reindex(columns=trained_columns, fill_value=0)

# 5. Predict immediately!
predicted_price = loaded_model.predict(new_house_ready)
print(f"The estimated price is: {predicted_price[0]:.2f} Lakhs")

The estimated price is: 147.02 Lakhs


In [ ]:

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
import joblib
import random

# ==========================================
# STEP 1: LOAD / CREATE DATA
# (Simulating 5,000 rows so this script runs instantly)
# ==========================================
np.random.seed(42)
cities = ['Mumbai', 'Delhi', 'Bangalore', 'Chennai']
prop_types = ['Apartment', 'Independent House', 'Villa']

data = {
    'City': [random.choice(cities) for _ in range(5000)],
    'Property_Type': [random.choice(prop_types) for _ in range(5000)],
    'BHK': np.random.randint(1, 6, 5000),
    'Bathrooms': np.random.randint(1, 5, 5000)
}
df = pd.DataFrame(data)

# Simulating logical prices based on features (Lakhs)
df['Price_Lakhs'] = (df['BHK'] * 25) + (df['Bathrooms'] * 10) + np.random.randint(10, 50, 5000)
df.loc[df['City'] == 'Mumbai', 'Price_Lakhs'] += 50
df.loc[df['Property_Type'] == 'Villa', 'Price_Lakhs'] += 40


# ==========================================
# STEP 2: FEATURE ENGINEERING
# ==========================================
# Map climates based on the city
climate_map = {
    'Mumbai': 'Tropical / Humid',
    'Chennai': 'Tropical / Humid',
    'Delhi': 'Composite / Extreme Hot & Cold',
    'Bangalore': 'Moderate / Tropical Savanna'
}
df['Climate'] = df['City'].map(climate_map)

# Add custom features
df['Total_Rooms'] = df['BHK']
df['Toilets'] = df['Bathrooms']
df['Kitchens'] = df['BHK'].apply(lambda x: 2 if x >= 4 else 1)
df['Max_People'] = (df['BHK'] * 2) + 1


# ==========================================
# STEP 3: PREPROCESSING & ENCODING
# ==========================================
# Separate the target (Price) from the features
X = df.drop('Price_Lakhs', axis=1)
y = df['Price_Lakhs']

# Convert text columns (City, Climate, Property_Type) into numbers
X_encoded = pd.get_dummies(X)


# ==========================================
# STEP 4: MODEL TRAINING & EVALUATION
# ==========================================
# Split data into 80% training and 20% testing
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42)

print("Training the Random Forest model... (This takes a moment)")
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Check model accuracy on the test set
predictions = model.predict(X_test)
mae = mean_absolute_error(y_test, predictions)
print(f"Model Accuracy: Predictions are off by an average of ~{mae:.2f} Lakhs.\n")


# ==========================================
# STEP 5: SAVE THE MODEL (DEPLOYMENT PREP)
# ==========================================
joblib.dump(model, 'house_price_predictor.pkl')
joblib.dump(X_encoded.columns, 'model_columns.pkl')
print("Model and columns successfully saved to disk as .pkl files!\n")


# ==========================================
# STEP 6: LOAD & PREDICT A BRAND NEW HOUSE
# ==========================================
# 1. Define the features of a new house
new_house = {
    'City': ['Delhi'],
    'Property_Type': ['Independent House'],
    'BHK': [4],
    'Bathrooms': [4],
    'Climate': ['Composite / Extreme Hot & Cold'],
    'Total_Rooms': [4],
    'Toilets': [4],
    'Kitchens': [2],
    'Max_People': [9]
}

# 2. Convert to DataFrame and Encode
new_house_df = pd.DataFrame(new_house)
new_house_encoded = pd.get_dummies(new_house_df)

# 3. Load the saved columns and align the new data
trained_columns = joblib.load('model_columns.pkl')
new_house_ready = new_house_encoded.reindex(columns=trained_columns, fill_value=0)

# 4. Load the saved model and predict
loaded_model = joblib.load('house_price_predictor.pkl')
predicted_price = loaded_model.predict(new_house_ready)

print("--- FINAL REAL-WORLD PREDICTION ---")
print(f"The estimated price for this 4 BHK Independent House in Delhi is: {predicted_price[0]:.2f} Lakhs")

Training the Random Forest model... (This takes a moment)
Model Accuracy: Predictions are off by an average of ~10.09 Lakhs.

Model and columns successfully saved to disk as .pkl files!

--- FINAL REAL-WORLD PREDICTION ---
The estimated price for this 4 BHK Independent House in Delhi is: 168.39 Lakhs


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
import joblib

# 1. DATA GENERATION
np.random.seed(42)
num_rows = 5000
cities = ['Mumbai', 'Delhi', 'Bangalore', 'Chennai', 'Kolkata', 'Dehradun']
property_types = ['Apartment', 'Independent House', 'Villa']
city_base_price = {'Mumbai': 120, 'Delhi': 85, 'Bangalore': 70, 'Chennai': 55, 'Kolkata': 40, 'Dehradun': 45}

city_col = np.random.choice(cities, num_rows, p=[0.25, 0.2, 0.2, 0.15, 0.1, 0.1])
prop_type_col = np.random.choice(property_types, num_rows, p=[0.7, 0.2, 0.1])
bhk_col = np.random.choice([1, 2, 3, 4, 5], num_rows, p=[0.1, 0.4, 0.3, 0.15, 0.05])
bath_col = np.clip(bhk_col + np.random.randint(-1, 2, size=num_rows), 1, 6)

prices = []
for i in range(num_rows):
    city, prop, bhk, bath = city_col[i], prop_type_col[i], bhk_col[i], bath_col[i]
    price = city_base_price[city] + (bhk * 20) + (bath * 8)
    if prop == 'Villa': price *= 2.0
    elif prop == 'Independent House': price *= 1.4
    price *= np.random.uniform(0.85, 1.15)
    prices.append(round(price, 2))

df = pd.DataFrame({'City': city_col, 'Property_Type': prop_type_col, 'BHK': bhk_col, 'Bathrooms': bath_col, 'Price_in_Lakhs': prices})

# 2. FEATURE ENGINEERING
climate_zones = {
    'Mumbai': 'Humid Tropical', 'Delhi': 'Composite / Extreme Hot & Cold',
    'Bangalore': 'Moderate / Tropical Savanna', 'Chennai': 'Tropical Wet & Dry',
    'Kolkata': 'Tropical Wet', 'Dehradun': 'Subtropical Highland / Cool'
}
df['Climate'] = df['City'].map(climate_zones).fillna('Warm & Humid')
df['Total_Rooms'] = df['BHK']
df['Toilets'] = df['Bathrooms']
df['Kitchens'] = df['Property_Type'].apply(lambda x: 2 if x in ['Independent House', 'Villa'] else 1)
df['Max_People'] = (df['BHK'] * 2) + 1

# 3. PREPROCESSING & ENCODING
df_encoded = pd.get_dummies(df, columns=['City', 'Property_Type', 'Climate'], drop_first=True)
y = df_encoded['Price_in_Lakhs']
X = df_encoded.drop('Price_in_Lakhs', axis=1)

# 4. TRAINING & EVALUATION
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

predictions = model.predict(X_test)
error = mean_absolute_error(y_test, predictions)
print(f"Model Training Complete. Mean Absolute Error: {error:.2f} Lakhs")

# 5. SAVE MODEL
joblib.dump(model, 'house_price_predictor.pkl')
joblib.dump(X.columns, 'model_columns.pkl')
print("Model and training columns saved successfully.")

Model Training Complete. Mean Absolute Error: 13.84 Lakhs
Model and training columns saved successfully.


In [ ]:
!pip install gradio

In [ ]:
import gradio as gr
import numpy as np
import joblib # Changed from pickle
import pandas as pd # Needed for feature engineering and encoding

# 1. Load your saved ML model/pipeline
# The model was saved as 'house_price_predictor.pkl' and column names as 'model_columns.pkl'
try:
    model = joblib.load('house_price_predictor.pkl')
    trained_columns = joblib.load('model_columns.pkl')
except FileNotFoundError:
    raise FileNotFoundError("Model files 'house_price_predictor.pkl' or 'model_columns.pkl' not found. "
                            "Please ensure the model training cell (like cell faf645b3 or duHi3cvzsME9) has been run successfully.")

# 2. Define your prediction function, aligning with the model's features
def predict_price(city, property_type, bhk, bathrooms):
    # Recreate the feature engineering steps
    input_data = {
        'City': [city],
        'Property_Type': [property_type],
        'BHK': [bhk],
        'Bathrooms': [bathrooms]
    }
    df_single = pd.DataFrame(input_data)

    # Reapply climate mapping from training (based on cell faf645b3 or duHi3cvzsME9)
    climate_zones = {
        'Mumbai': 'Humid Tropical', 'Delhi': 'Composite / Extreme Hot & Cold',
        'Bangalore': 'Moderate / Tropical Savanna', 'Chennai': 'Tropical Wet & Dry',
        'Kolkata': 'Tropical Wet', 'Dehradun': 'Subtropical Highland / Cool'
    }
    df_single['Climate'] = df_single['City'].map(climate_zones).fillna('Warm & Humid')

    # Reapply other derived features from training
    df_single['Total_Rooms'] = df_single['BHK']
    df_single['Toilets'] = df_single['Bathrooms']
    df_single['Kitchens'] = df_single['Property_Type'].apply(lambda x: 2 if x in ['Independent House', 'Villa'] else 1)
    df_single['Max_People'] = (df_single['BHK'] * 2) + 1

    # Encode categorical features exactly as done during training
    df_encoded_single = pd.get_dummies(df_single, columns=['City', 'Property_Type', 'Climate'], drop_first=True)

    # Align columns with the trained model's columns
    # This is crucial to handle cases where a specific category isn't present in the single input
    final_features = df_encoded_single.reindex(columns=trained_columns, fill_value=0)

    # Make the prediction
    prediction = model.predict(final_features)[0]

    return f"Predicted House Price: ₹ {prediction:,.2f} Lakhs"

# 3. Create the Gradio Interface with inputs matching the new predict_price function
# The choices for dropdowns and ranges for sliders are based on the training data generation
demo = gr.Interface(
    fn=predict_price,
    inputs=[
        gr.Dropdown(choices=['Mumbai', 'Delhi', 'Bangalore', 'Chennai', 'Kolkata', 'Dehradun'], label="City"),
        gr.Dropdown(choices=['Apartment', 'Independent House', 'Villa'], label="Property Type"),
        gr.Slider(1, 5, step=1, label="Number of BHK"), # BHK was 1-5 in training data
        gr.Slider(1, 6, step=1, label="Number of Bathrooms") # Bathrooms was 1-6 in training data
    ],
    outputs=gr.Textbox(label="Predicted Output"),
    title="India House Price Predictor"
)

# 4. Launch the app
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0b2d8e59257f58e629.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
